In [21]:
# ===========================================
# Cell 1: Google Drive Mount
# ===========================================
from google.colab import drive

drive.mount('/content/drive')
print("Drive mount done!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mount done!


In [35]:
# ===========================================
# Cell 2: Config + Path Setup
# ===========================================
import os
import yaml

CONFIG_PATH = "/content/config.yaml"

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

DRIVE_ROOT   = f"/content/drive/MyDrive/{config['project_name']}"
DATA_PATH    = os.path.join(DRIVE_ROOT, config['paths']['processed_data'])
MODEL_PATH   = os.path.join(DRIVE_ROOT, config['paths']['models'])
REPORT_PATH  = os.path.join(DRIVE_ROOT, config['paths']['outputs'], "reports")

os.makedirs(MODEL_PATH,  exist_ok=True)
os.makedirs(REPORT_PATH, exist_ok=True)

RANDOM_STATE = config['model']['random_state']   # 42
TEST_SIZE    = config['model']['test_size']       # 0.2

print(f"Project : {config['project_name']}")
print(f"Data    : {DATA_PATH}")
print(f"Models  : {MODEL_PATH}")

Project : EcoTracing
Data    : /content/drive/MyDrive/EcoTracing/data/processed
Models  : /content/drive/MyDrive/EcoTracing/models


In [36]:
# ===========================================
# Cell 3: Library Import + EnergyMLP Class 
# ===========================================
import numpy as np
import pandas as pd
import pickle
import json
import time
import torch
import torch.nn as nn
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler


class EnergyMLP(nn.Module):
    """
    저장된 state_dict 구조에 맞춘 EnergyMLP
    - 속성명 : network (net 아님)
    - BatchNorm1d : 각 hidden layer 뒤에 포함
    - 구조 : Linear -> BatchNorm1d -> ReLU (x hidden_sizes 수) -> Linear

    Input : cpu, memory, duration (3개)
    Output : energy_kwh (1개)
    """
    def __init__(self, input_size=3, hidden_sizes=[512, 256, 128], dropout=0.0):
        super().__init__()
        layers = []
        prev = input_size
        for h in hidden_sizes:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),  # 저장된 모델에 BN 포함되어 있음
                nn.ReLU()
            ]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.network = nn.Sequential(*layers)  # net -> network

    def forward(self, x):
        return self.network(x).squeeze(1)


print("Libraries loaded!")

Libraries loaded!


In [40]:
# ===========================================
# Cell 4: Load Data + Split
# ===========================================
parquet_path = os.path.join(DATA_PATH, "instance_usage_full_processed.parquet")
df = pd.read_parquet(parquet_path)
print(f"Total rows: {len(df):,}")

# RF features (4개) / MLP features (3개) / target
RF_FEATURES = ['cpu', 'memory', 'duration']
MLP_FEATURES = ['cpu', 'memory', 'duration']
TARGET = 'energy_kwh'

X = df[RF_FEATURES] # RF 기준으로 split, MLP는 인덱스 슬라이싱
y = df[TARGET]

# 80% train, 20% test (original 학습과 동일한 random_state)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# test를 meta_val(50%) / final_test(50%)로 추가 분할
# meta_val  : RF/MLP 예측값으로 Meta LR 학습
# final_test: 최종 성능 평가
X_meta, X_final, y_meta, y_final = train_test_split(
    X_test, y_test, test_size=0.5, random_state=RANDOM_STATE
)


print(f"X_meta : {X_meta.shape}")
print(f"X_final : {X_final.shape}")

Total rows: 19,523,808
X_meta : (1952381, 3)
X_final : (1952381, 3)


In [41]:
# ===========================================
# Cell 5: Load Base Models (RF + MLP)
# ===========================================

# --- RF 로드 ---
rf_file = os.path.join(MODEL_PATH, config['model']['model_names']['randomforest'])
with open(rf_file, 'rb') as f:
    rf = pickle.load(f)
print(f"RF loaded: {rf_file}")

# --- MLP 로드 ---
mlp_file = os.path.join(MODEL_PATH, config['model']['model_names']['mlp'])
scaler_x_file = os.path.join(MODEL_PATH, config['model']['model_names']['scaler_x'])
scaler_y_file = os.path.join(MODEL_PATH, config['model']['model_names']['scaler_y'])

mlp = EnergyMLP(input_size=3, hidden_sizes=[512, 256, 128], dropout=0.0)
mlp.load_state_dict(torch.load(mlp_file, map_location='cpu'))
mlp.eval()

with open(scaler_x_file, 'rb') as f:
    scaler_X = pickle.load(f)
with open(scaler_y_file, 'rb') as f:
    scaler_y = pickle.load(f)

print(f"MLP loaded: {mlp_file}")


def predict_mlp(model, scaler_X, scaler_y, X_df):
    """
    MLP로 에너지 예측
    Args:
        model : EnergyMLP 인스턴스
        scaler_X : 입력 StandardScaler (3 features)
        scaler_y : 출력 StandardScaler
        X_df : DataFrame (cpu, memory, duration 컬럼 포함)
    Returns:
        np.array: 예측된 energy_kwh
    """
    X_np = X_df[MLP_FEATURES].values.astype(np.float32)
    X_sc = scaler_X.transform(X_np)
    X_t = torch.tensor(X_sc, dtype=torch.float32)
    with torch.no_grad():
        y_sc = model(X_t).numpy().reshape(-1, 1)
    return scaler_y.inverse_transform(y_sc).ravel()

RF loaded: /content/drive/MyDrive/EcoTracing/models/energy_model_randomforest.pkl
MLP loaded: /content/drive/MyDrive/EcoTracing/models/energy_model_mlp.pt


In [42]:
# ===========================================
# Cell 6: Meta Train 예측값 생성
# ===========================================
# Meta LR 학습용 feature = [RF_pred, MLP_pred]

print("Generating meta_train predictions...")
rf_pred_meta  = rf.predict(X_meta[RF_FEATURES])
mlp_pred_meta = predict_mlp(mlp, scaler_X, scaler_y, X_meta)

# Meta feature matrix
X_meta_stack = np.column_stack([rf_pred_meta, mlp_pred_meta])
print(f"X_meta_stack shape: {X_meta_stack.shape}")  # (n, 2)


Generating meta_train predictions...
X_meta_stack shape: (1952381, 2)


In [43]:
# ===========================================
# Cell 7: Meta LR 학습
# ===========================================
meta_lr = LinearRegression()
meta_lr.fit(X_meta_stack, y_meta.values)

print(f"Meta LR coef: RF={meta_lr.coef_[0]:.4f}, MLP={meta_lr.coef_[1]:.4f}")
print(f"Meta LR intercept: {meta_lr.intercept_:.6f}")

Meta LR coef: RF=1.0017, MLP=-0.0017
Meta LR intercept: -0.000000


In [44]:
# ===========================================
# Cell 8: Final Test 평가
# ===========================================
rf_pred_final  = rf.predict(X_final[RF_FEATURES])
mlp_pred_final = predict_mlp(mlp, scaler_X, scaler_y, X_final)

X_final_stack  = np.column_stack([rf_pred_final, mlp_pred_final])
y_pred_final   = meta_lr.predict(X_final_stack)

y_true = y_final.values
rmse = np.sqrt(mean_squared_error(y_true, y_pred_final))
mae  = mean_absolute_error(y_true, y_pred_final)
r2   = r2_score(y_true, y_pred_final)
mape = np.mean(np.abs((y_true - y_pred_final) / (y_true + 1e-10))) * 100

print(f"\n[MLP + RF Stacking 성능]")
print(f"  RMSE : {rmse:.8f} kWh")
print(f"  MAE  : {mae:.8f} kWh")
print(f"  R2   : {r2:.8f}")
print(f"  MAPE : {mape:.4f}%")

# 단일 모델 비교
print(f"\n[단일 모델 MAPE 비교]")
mape_rf  = np.mean(np.abs((y_true - rf_pred_final)  / (y_true + 1e-10))) * 100
mape_mlp = np.mean(np.abs((y_true - mlp_pred_final) / (y_true + 1e-10))) * 100
print(f"  RF   MAPE : {mape_rf:.4f}%")
print(f"  MLP  MAPE : {mape_mlp:.4f}%")
print(f"  Stack MAPE: {mape:.4f}%")


[MLP + RF Stacking 성능]
  RMSE : 0.00001318 kWh
  MAE  : 0.00000761 kWh
  R2   : 0.99999558
  MAPE : 0.0632%

[단일 모델 MAPE 비교]
  RF   MAPE : 0.0514%
  MLP  MAPE : 5.7961%
  Stack MAPE: 0.0632%


In [45]:
# ===========================================
# Cell 9: Streamlit 조건 테스트
# ===========================================
# 기준값: CPU=0.5, Memory=0.3, 1시간 -> 공식 = 0.365 kWh
import datetime

FORMULA_KWH = 0.365
TEST_CPU = 0.5
TEST_MEM = 0.3
TEST_DUR_H = 1.0
TEST_DUR_SEC = TEST_DUR_H * 3600
TEST_HOUR = datetime.datetime.now().hour

# RF 예측
rf_input = np.array([[TEST_CPU, TEST_MEM, TEST_DUR_SEC, TEST_HOUR]])
rf_out = rf.predict(rf_input)[0]

# MLP 예측
mlp_input_df = pd.DataFrame([[TEST_CPU, TEST_MEM, TEST_DUR_SEC]], columns=MLP_FEATURES)
mlp_out = predict_mlp(mlp, scaler_X, scaler_y, mlp_input_df)[0]

# Stacking 예측
stack_input = np.array([[rf_out, mlp_out]])
stack_out = meta_lr.predict(stack_input)[0]

error_pct = abs(stack_out - FORMULA_KWH) / FORMULA_KWH * 100

print(f"\n[Streamlit 조건 테스트] CPU=50%, Mem=30%, 1시간")
print(f" 공식 기준값 : {FORMULA_KWH:.4f} kWh")
print(f" RF 예측 : {rf_out:.6f} kWh")
print(f" MLP 예측 : {mlp_out:.6f} kWh")
print(f" Stacking 예측: {stack_out:.6f} kWh")
print(f" 공식 대비 오차: {error_pct:.2f}%")
print(f"\n 이전 MLP 단독 오차: 13.2%]")


# ===========================================
# Cell 10: 모델 저장 + 결과 저장
# ===========================================

# Meta LR 저장
stack_file = os.path.join(MODEL_PATH, config['model']['model_names']['stacking_mlp_rf'])
with open(stack_file, 'wb') as f:
    pickle.dump(meta_lr, f)
print(f"Saved: {stack_file}")

# 결과 저장
results = {
    'method' : 'MLP + RF Stacking (Meta LinearRegression)',
    'base_models' : {
        'rf' : config['model']['model_names']['randomforest'],
        'mlp': config['model']['model_names']['mlp'],
    },
    'meta_model' : 'LinearRegression',
    'meta_coef' : {'RF': float(meta_lr.coef_[0]), 'MLP': float(meta_lr.coef_[1])},
    'meta_intercept' : float(meta_lr.intercept_),
    'rf_features' : RF_FEATURES,
    'mlp_features' : MLP_FEATURES,
    'metrics' : {
        'rmse': float(rmse),
        'mae' : float(mae),
        'r2' : float(r2),
        'mape': float(mape),
    },
    'streamlit_test' : {
        'cpu' : TEST_CPU,
        'memory' : TEST_MEM,
        'duration_h' : TEST_DUR_H,
        'formula_kwh' : FORMULA_KWH,
        'rf_pred' : float(rf_out),
        'mlp_pred' : float(mlp_out),
        'stack_pred' : float(stack_out),
        'error_pct' : float(error_pct),
    }
}

result_file = os.path.join(REPORT_PATH, config['model']['results']['results_json_stacking'])
with open(result_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Results saved: {result_file}")
print("Done!")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


ValueError: X has 4 features, but RandomForestRegressor is expecting 3 features as input.